## импорты

In [1]:
import numpy as np
import pandas as pd

## вспомогательная функция

In [2]:
def quantile(values, q):
    v = np.sort(values[~np.isnan(values)])
    if v.size == 0:
        return np.nan
    pos = (v.size - 1) * q
    lo = int(np.floor(pos))
    hi = min(lo + 1, v.size - 1)
    return v[lo] + (v[hi] - v[lo]) * (pos - lo)

## Target encoding



In [3]:
class TargetEncoder:
    def __init__(self, cols, smoothing=10.0):
        self.cols = list(cols)
        self.smoothing = float(smoothing)

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.prior_ = y.mean()
        self.maps_ = {}
        m = self.smoothing
        for col in self.cols:
            df = pd.DataFrame({"k": X[col].astype(str).to_numpy(), "y": y})
            g = df.groupby("k")["y"].agg(["sum", "count"])
            self.maps_[col] = (g["sum"] + m * self.prior_) / (g["count"] + m)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col] = X[col].astype(str).map(self.maps_[col]).astype(float).fillna(self.prior_)
        return X

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

## Winsorizing



In [4]:
class Winsorizer:
    def __init__(self, cols, lower=0.01, upper=0.99):
        self.cols = list(cols)
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.bounds_ = {}
        for col in self.cols:
            v = X[col].to_numpy(dtype=float)
            self.bounds_[col] = (quantile(v, self.lower), quantile(v, self.upper))
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            lo, hi = self.bounds_[col]
            v = X[col].to_numpy(dtype=float)
            nan = np.isnan(v)
            v = np.where(v < lo, lo, np.where(v > hi, hi, v))
            v[nan] = np.nan
            X[col] = v
        return X

    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)

## KNN


In [5]:
class KNNImputer:
    def __init__(self, cols, n_neighbors=5):
        self.cols = list(cols)
        self.k = n_neighbors

    def fit(self, X, y=None):
        A = X[self.cols].to_numpy(dtype=float)
        self.mu_ = np.nanmean(A, axis=0)
        sd = np.nanstd(A, axis=0)
        self.sd_ = np.where(sd > 0, sd, 1.0)
        self.raw_ = A
        self.Z_ = (A - self.mu_) / self.sd_
        return self

    @staticmethod
    def nan_euclidean(P, Q):
        mp, mq = ~np.isnan(P), ~np.isnan(Q)
        P0, Q0 = np.where(mp, P, 0.0), np.where(mq, Q, 0.0)
        mpf, mqf = mp.astype(float), mq.astype(float)
        sq = (P0 ** 2) @ mqf.T + mpf @ (Q0 ** 2).T - 2.0 * P0 @ Q0.T
        sq = np.maximum(sq, 0.0)
        common = mpf @ mqf.T
        with np.errstate(divide="ignore", invalid="ignore"):
            d = np.sqrt(sq * P.shape[1] / common)
        d[common == 0] = np.inf
        return d

    def transform(self, X):
        X = X.copy()
        A = X[self.cols].to_numpy(dtype=float)
        miss_rows = np.where(np.isnan(A).any(axis=1))[0]
        if miss_rows.size == 0:
            return X
        D = self.nan_euclidean((A[miss_rows] - self.mu_) / self.sd_, self.Z_)
        out = A.copy()
        for j in range(A.shape[1]):
            need = np.isnan(A[miss_rows, j])
            if not need.any():
                continue
            Dj = D[need].copy()
            Dj[:, np.isnan(self.raw_[:, j])] = np.inf
            idx = np.argsort(Dj, axis=1, kind="stable")[:, :self.k]
            valid = np.isfinite(np.take_along_axis(Dj, idx, axis=1))
            vals = np.where(valid, self.raw_[idx, j], 0.0)
            cnt = valid.sum(axis=1)
            imp = np.where(cnt > 0, vals.sum(axis=1) / np.maximum(cnt, 1), self.mu_[j])
            out[miss_rows[need], j] = imp
        X[self.cols] = out
        return X

    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)

## данные



In [6]:
NUM = ["pclass", "age", "sibsp", "parch", "fare"]
WIN = ["age", "fare"]
CAT = ["sex", "embarked", "pclass"]

df = pd.read_csv("data/titanic.csv")
X = df[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()
y = df["survived"].to_numpy()

perm = np.random.default_rng(0).permutation(len(X))
cut = int(0.8 * len(X))
X_tr, X_te = X.iloc[perm[:cut]].reset_index(drop=True), X.iloc[perm[cut:]].reset_index(drop=True)
y_tr, y_te = y[perm[:cut]], y[perm[cut:]]

print(f"train {len(X_tr)}, test {len(X_te)}")
print("пропуски в train:", X_tr.isna().sum()[X_tr.isna().sum() > 0].to_dict())
X_tr.head()

train 1047, test 262
пропуски в train: {'age': 210, 'fare': 1, 'embarked': 2}


,pclass,sex,age,sibsp,parch,fare,embarked
0,3,male,11.0,5,2,46.900,S
1,3,male,NaN,0,0,7.225,C
2,1,female,24.0,0,0,69.300,C
3,3,female,33.0,1,2,20.575,S
4,2,female,29.0,1,0,26.000,S


## тестим KNN

In [7]:
knn = KNNImputer(NUM, n_neighbors=5).fit(X_tr)
X_tr_knn, X_te_knn = knn.transform(X_tr), knn.transform(X_te)

miss = X_tr["age"].isna()
print("пропусков age: до", miss.sum(), "-> после", X_tr_knn["age"].isna().sum(), "(train),",
      X_te["age"].isna().sum(), "->", X_te_knn["age"].isna().sum(), "(test)")
pd.DataFrame({"pclass": X_tr.loc[miss, "pclass"], "sibsp": X_tr.loc[miss, "sibsp"],
              "parch": X_tr.loc[miss, "parch"], "fare": X_tr.loc[miss, "fare"],
              "age до": X_tr.loc[miss, "age"], "age после": X_tr_knn.loc[miss, "age"].round(1)}).head(8)

пропусков age: до 210 -> после 0 (train), 53 -> 0 (test)


,pclass,sibsp,parch,fare,age до,age после
1,3,0,0,7.2250,NaN,21.6
10,3,2,0,21.6792,NaN,24.0
11,3,0,0,6.9500,NaN,39.5
20,1,0,0,227.5250,NaN,35.3
21,1,0,0,26.5500,NaN,53.4
22,3,0,0,7.2292,NaN,26.0
29,3,0,0,7.8875,NaN,32.3
31,3,0,0,7.8958,NaN,25.2


## тестим Winsorizing

In [8]:
win = Winsorizer(WIN, lower=0.01, upper=0.99).fit(X_tr_knn)
X_tr_w, X_te_w = win.transform(X_tr_knn), win.transform(X_te_knn)

pd.DataFrame({c: {"нижняя граница": win.bounds_[c][0], "верхняя граница": win.bounds_[c][1],
                  "min до": X_tr_knn[c].min(), "max до": X_tr_knn[c].max(),
                  "min после": X_tr_w[c].min(), "max после": X_tr_w[c].max(),
                  "обрезано, %": 100 * ((X_tr_knn[c] < win.bounds_[c][0]) | (X_tr_knn[c] > win.bounds_[c][1])).mean()}
              for c in WIN}).T.round(2)

,нижняя граница,верхняя граница,min до,max до,min после,max после,"обрезано, %"
age,1.0,64.54,0.17,76.00,1.0,64.54,1.91
fare,0.0,255.54,0.00,512.33,0.0,255.54,1.05


## тестим Target encoding

In [ ]:
for m in [0, 10, 100]:
    te = TargetEncoder(["embarked"], smoothing=m).fit(X_tr_w, y_tr)
    print(f"m={m:<3}", te.maps_["embarked"].round(3).to_dict())

counts = X_tr_w["embarked"].astype(str).value_counts().to_dict()
print("размер категорий:", counts, "| prior =", round(y_tr.mean(), 3))

m=0   {'C': 0.562, 'Q': 0.375, 'S': 0.342}
m=10  {'C': 0.554, 'Q': 0.377, 'S': 0.343}
m=100 {'C': 0.509, 'Q': 0.384, 'S': 0.348}
размер категорий: {'S': 722, 'C': 219, 'Q': 104} | prior = 0.393


In [ ]:
te = TargetEncoder(CAT, smoothing=10).fit(X_tr_w, y_tr)
X_tr_f, X_te_f = te.transform(X_tr_w), te.transform(X_te_w)
X_te_f.head()

,pclass,sex,age,sibsp,parch,fare,embarked
0,0.443644,0.733487,60.0,1.0,0.0,26.000,0.342794
1,0.258047,0.201645,28.0,0.0,0.0,22.525,0.342794
2,0.258047,0.201645,24.0,0.0,0.0,9.500,0.342794
3,0.643657,0.201645,40.6,0.0,0.0,26.000,0.342794
4,0.258047,0.201645,50.0,1.0,0.0,14.500,0.342794
